In [13]:
import yfinance as yf
import pandas as pd
import sqlite3

# ==================== EXTRACT ====================
tickers = ['AAPL', 'MSFT', 'GOOGL', 'AMZN']
df = yf.download(tickers, period='1mo', interval='1d')
df = df.stack(level=1).reset_index()


[*********************100%***********************]  4 of 4 completed


In [14]:
# ==================== TRANSFORM ====================
# Dicionário de renomeação
rename_map = {
    'Date': 'data',
    'Open': 'abertura',
    'High': 'maxima',
    'Low': 'minima',
    'Close': 'fechamento',
    'Adj Close': 'fechamento_ajustado',
    'Volume': 'total'
}
# Renomeia apenas as colunas que existem
df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns}, inplace=True)

# Cria média móvel de 7 dias (usando 'fechamento' que deve existir)
df['media_movel_7d'] = df['fechamento'].rolling(window=7).mean()
df.dropna(inplace=True)

# Define as colunas de preços (inclui apenas as que existem no DataFrame)
colunas_precos = ['abertura', 'maxima', 'minima', 'fechamento', 'fechamento_ajustado', 'total', 'media_movel_7d']
colunas_existentes = [col for col in colunas_precos if col in df.columns]

# Formata as colunas existentes
df[colunas_existentes] = df[colunas_existentes].astype(int).round(2)

df['total'] = (df['total'] / 1_000_000).round(3)


In [15]:
display(df.head())

Price,data,Ticker,fechamento,maxima,minima,abertura,total,media_movel_7d
6,2026-06-23,GOOGL,346,349,340,340,34.008,303
7,2026-06-23,MSFT,373,377,370,372,40.648,314
8,2026-06-24,AAPL,293,299,292,295,53.082,322
9,2026-06-24,AMZN,234,242,232,233,70.283,306
10,2026-06-24,GOOGL,345,353,341,349,44.997,303


In [16]:
# ==================== LOAD ====================
conn = sqlite3.connect('dados_financeiros.db')
df.to_sql('acoes', conn, if_exists='replace', index=False)
conn.close()
print("✅ Dados carregados com sucesso!")

✅ Dados carregados com sucesso!
